# Choropleth

`mapclassify` is intended to be used with visualization packages to handle the actual rendering of the choropleth maps defined on its classifiers. In this notebook, we explore some examples of how this is done. The notebook also includes an example that combines `mapclassify` with [ipywidgets](https://ipywidgets.readthedocs.io/en/latest/) to allow for the interactive exploration of the choice of:

- classification method
- number of classes
- colormap

In [ ]:
import geopandas
import libpysal
import matplotlib.pyplot as plt

import mapclassify

mapclassify.__version__

The example in this notebook uses data on sudden death infant syndrome for counties in North Carolina. It is a built-in dataset  available through `libpysal`. We use `libpysal` to obtain the path to the shapefile and then use `geopandas` to create a geodataframe from the shapefile:

In [ ]:
libpysal.examples.explain("sids2")

In [ ]:
pth = libpysal.examples.get_path("sids2.shp")
gdf = geopandas.read_file(pth)
gdf.head()

Once created, the geodataframe has a `plot` method that can be called to create our first choropleth map. We will specify the column to classify and plot as `SIDR79`: SIDS death rate per 1,000 births (1979-84). The classification scheme is set to  `Quantiles`, and the number of classes set to `k=10` (declies):

In [ ]:
ax = gdf.plot(column="SIDR79", scheme="Quantiles", k=10, figsize=(16, 9))
ax.set_axis_off()

We can peak under the hood a bit and recreate the classification object that was used in the previous choropleth:

In [ ]:
q10 = mapclassify.Quantiles(gdf.SIDR79, k=10)
q10

For quick, exploratory work, the classifier object has its own `plot` method that takes a geodataframe as an argument:

In [ ]:
q10.plot(gdf);

Back to working directly with the dataframe, we can toggle on the `legend`:

In [ ]:
ax = gdf.assign(cl=q10.yb).plot(
    figsize=(16, 9),
    column="cl",
    categorical=True,
    k=10,
    cmap="OrRd",
    linewidth=0.1,
    edgecolor="white",
    legend=True,
)
ax.set_axis_off()

Here we see the 10 classes, but without more specific information on the legend, the user has to know that 0 is the first decile and 9 the 10th. We also do not know the values that define these classes. 

We can rectify this as follows:

In [ ]:
q10.get_legend_classes()

In [ ]:
mapping = {i: s for i, s in enumerate(q10.get_legend_classes())}
mapping

In [ ]:
def replace_legend_items(legend, mapping):
    for txt in legend.texts:
        for k, v in mapping.items():
            if txt.get_text() == str(k):
                txt.set_text(v)

In [ ]:
ax = gdf.assign(cl=q10.yb).plot(
    figsize=(16, 9),
    column="cl",
    categorical=True,
    k=10,
    cmap="OrRd",
    linewidth=0.1,
    edgecolor="white",
    legend=True,
    legend_kwds={"loc": "lower right"},
)
ax.set_axis_off()
replace_legend_items(ax.get_legend(), mapping)

## Interactive Exploration of Choropleth Classification

Next, we develop a small application that relies on `mapclassify` together with [palettable](https://jiffyclub.github.io/palettable/) and [ipywidgets](https://ipywidgets.readthedocs.io/en/latest/) to explore the choice of:

- classification method
- number of classes
- colormap



In [ ]:
from ipywidgets import (
    Button,
    Dropdown,
    FloatSlider,
    HBox,
    IntSlider,
    Label,
    Output,
    RadioButtons,
    Tab,
    VBox,
    interact,
)
from palettable import colorbrewer

Declare that our application shall have 3 options for color scheme...

In [ ]:
data_type = RadioButtons(options=["Sequential", "Diverging", "Qualitative"])

... and define thise 3 colormaps based on [colorbrewer](https://colorbrewer2.org/#type=sequential&scheme=BuGn&n=3).

In [ ]:
sequential = colorbrewer.COLOR_MAPS["Sequential"]
diverging = colorbrewer.COLOR_MAPS["Diverging"]
qualitative = colorbrewer.COLOR_MAPS["Qualitative"]

Next, create value and name options to bind the radio button to the dropdown menus...

In [ ]:
bindings = {
    "Sequential": range(3, 9 + 1),
    "Diverging": range(3, 11 + 1),
    "Qualitative": range(3, 12 + 1),
}
cmap_bindings = {
    "Sequential": list(sequential.keys()),
    "Diverging": list(diverging.keys()),
    "Qualitative": list(qualitative.keys()),
}
class_val = Dropdown(options=bindings[data_type.value], value=5)
cmap_val = Dropdown(options=cmap_bindings[data_type.value])

... and define 7 `mapclassify` objects data classification. 

In [ ]:
k_classifiers = {
    "equal_interval": mapclassify.EqualInterval,
    "fisher_jenks": mapclassify.FisherJenks,
    "jenks_caspall": mapclassify.JenksCaspall,
    "jenks_caspall_forced": mapclassify.JenksCaspallForced,
    "maximum_breaks": mapclassify.MaximumBreaks,
    "natural_breaks": mapclassify.NaturalBreaks,
    "quantiles": mapclassify.Quantiles,
}

Now it all together with helper functions for visualizing & updating plot options.

In [ ]:
def k_values(ctype, cmap):
    k = list(colorbrewer.COLOR_MAPS[ctype][cmap].keys())
    return list(map(int, k))


def update_map(method="quantiles", k=5, cmap="Blues"):
    classifier = k_classifiers[method](gdf.SIDR79, k=k)
    mapping = {i: s for i, s in enumerate(classifier.get_legend_classes())}
    plt_kws = {
        "figsize": (16, 9),
        "column": "cl",
        "categorical": True,
        "legend": True,
        "lw": 0.1,
    }
    ax = gdf.assign(cl=classifier.yb).plot(
        k=k, cmap=cmap, ec="grey", legend_kwds={"loc": "lower left"}, **plt_kws
    )
    ax.set_axis_off()
    ax.set_title("SIDR79")
    replace_legend_items(ax.get_legend(), mapping)


def type_change(change):
    class_val.options = bindings[change["new"]]
    cmap_val.options = cmap_bindings[change["new"]]


def cmap_change(change):
    cmap = change["new"]
    ctype = data_type.value
    k = k_values(ctype, cmap)
    class_val.options = k

Finally, let's play with the interactive plot!
* *the plot will not render in the docs, but may be accessed directly by running this notebook locally or as a binder.*

In [ ]:
data_type.observe(type_change, names=["value"])
cmap_val.observe(cmap_change, names=["value"])
out = Output()
Tab().children = [out]
interact(update_map, method=list(k_classifiers.keys()), cmap=cmap_val, k=class_val)
display(VBox([data_type, out]))

Changing the type of colormap (sequential, diverging, qualitative) will update the options for the available color maps (`cmap`). Changing any of the values using the dropdowns will update the classification and the resulting choropleth map.

It is important to note that the example variable is best portrayed with the sequential colormaps. The other two types of colormaps are included for demonstration purposes only.